# V14 – Betriebssysteme Teil 2: Theorie

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- den Unterschied zwischen **Prozess** und **Thread** präzise erklären und Beispiele aus dem Ingenieurs-Alltag zuordnen,
- den **Prozess-Lebenszyklus** mit den Zuständen *Neu → Bereit → Laufend → Wartend → Beendet* beschreiben,
- die wichtigsten **Scheduling-Strategien** (FCFS, SJF, Round-Robin, Prioritäts-Scheduling) benennen und einordnen, wann welche sinnvoll ist,
- die Grundidee von **virtuellem Speicher** und **Paging** nachvollziehen und verstehen, was ein Page Fault ist,
- die **Speicherhierarchie** (Register → Cache → RAM → SSD → HDD) in ihrer Größenordnung einordnen,
- die Struktur eines **hierarchischen Dateisystems** und die Bedeutung der **rwx-Rechte** erklären.

## ⏱️ Zeitbudget
Ca. 60 Minuten Lesezeit. Nimm dir Zeit für die Diagramme — sie sind der eigentliche Inhalt, der Text ist nur die Begleitmusik.

## 🧭 TL;DR
Das Betriebssystem jongliert für dich tausende **Prozesse**, teilt CPU-Zeit über einen **Scheduler** zu, simuliert jedem Programm einen eigenen Adressraum über **virtuellen Speicher** und organisiert alle persistenten Daten in einem **Dateisystem-Baum**. Als Ingenieurin oder Ingenieur triffst du diese Konzepte überall, wo mehrere Messungen, Simulationen oder Steuerungen parallel laufen müssen — vom Hintergrund-Logger über den FEM-Solver auf 16 Kernen bis zum Web-Dashboard, das deine Produktionslinie überwacht.

## 📦 Voraussetzungen
- V13 — Betriebssysteme & Rechnerarchitektur Teil 1 (CPU, RAM, von-Neumann-Architektur)
- V00-Recap dieses Ordners (Schleifensteuerung, List Comprehensions, Listen)


## 0. Warum brauchen wir ein Betriebssystem?

In V13 hast du die Architektur eines Rechners auf der Hardware-Ebene betrachtet: eine **CPU**, die Instruktionen abarbeitet, ein **RAM** als flüchtiger Arbeitsspeicher, eine **SSD** für persistente Daten sowie die Busse, die beides verbinden. Diese Hardware allein ist aber nicht benutzbar. Sie weiß nichts davon, dass du gleichzeitig einen Python-Datenlogger, einen Browser und einen FEM-Solver laufen lassen willst, und sie hat keine Ahnung, wie aus dem flachen Bytestrom einer SSD eine Datei mit dem Namen `messung_03.csv` werden soll.

Genau diese Lücke schließt das **Betriebssystem** (kurz **OS** für *Operating System*). Es ist die Schicht zwischen Hardware und Anwendung, und es erfüllt im Kern vier Aufgaben: Es verteilt die CPU-Zeit auf alle laufenden Programme, verwaltet den Arbeitsspeicher, bietet ein strukturiertes Dateisystem und steuert den Zugriff auf Ein-/Ausgabegeräte wie Netzwerk, Tastatur und Drucker. Für dich als Anwender fühlt es sich so an, als hätte jedes deiner Programme den Rechner für sich allein — und das ist kein Zufall, sondern das erklärte Designziel jedes modernen Betriebssystems.

In diesem Notebook betrachten wir die vier zentralen Konzepte, die diese Illusion möglich machen: **Prozesse**, **Threads**, **virtueller Speicher** und **Dateisystem**. Alle vier werden dir später in der Praxis wiederbegegnen — spätestens dann, wenn du deinen ersten Mess-Daemon im Hintergrund laufen lassen willst oder der RAM auf deinem Laptop beim nächsten ANSYS-Lauf knapp wird.


## 1. Prozesse — die ausführbare Einheit

Wenn du ein Programm startest — etwa `python messung.py` — erzeugt das Betriebssystem daraus einen **Prozess**. Ein Prozess ist nicht dasselbe wie das Programm auf der Festplatte: Das Programm ist eine Datei, ein Prozess ist eine konkrete *laufende Instanz* dieser Datei mit eigenen Ressourcen.

> [!NOTE]
> Ein **Prozess** ist eine in Ausführung befindliche Programm-Instanz. Er umfasst den geladenen Programmcode, seinen **eigenen Adressraum** im RAM, einen eigenen **Stack** für Funktionsaufrufe, offene Datei-Handles, Netzwerkverbindungen und eine eindeutige **Prozess-ID (PID)**.

Zwei Prozesse, die aus demselben Programm gestartet wurden, sind trotzdem vollkommen voneinander isoliert. Stürzt einer ab, läuft der andere unbeirrt weiter. Liest einer an Adresse `0x7fff00a8`, hat das mit dem Speicher des anderen nichts zu tun. Diese Isolation ist die wichtigste Sicherheitsgarantie, die ein modernes OS liefert.


### Ingenieur-Beispiel: der Datenlogger im Hintergrund

Stell dir vor, an deinem Prüfstand läuft ein Python-Skript `datenlogger.py`, das alle 100 ms einen Drehmoment-Sensor ausliest und in eine CSV schreibt. Parallel sitzt du selbst an der grafischen Oberfläche und wertest die Daten des Vortages in Jupyter aus. Dritter im Bunde ist vielleicht ein Webserver, der die aktuelle Messung als Live-Chart ins Intranet streamt.

Drei unabhängige **Prozesse**, drei verschiedene PIDs, drei separate Speicherbereiche. Dein Jupyter-Absturz bringt den Logger nicht zu Fall, und wenn du dem Webserver den Saft abdrehst, geht die Messung trotzdem weiter. Genau dafür ist die Prozess-Isolation da — und genau deswegen ist die Frage „wer läuft gerade als welcher Prozess?" eine der wichtigsten, die du dir in produktiven Umgebungen stellen kannst.


### 1.1 Der Prozess-Lebenszyklus

Ein Prozess durchläuft während seines Lebens mehrere Zustände. Wenn du ein Programm startest, entsteht er zunächst im Zustand **Neu**. Sobald alle Ressourcen geladen sind, wechselt er nach **Bereit** — er wartet darauf, dass der Scheduler ihm CPU-Zeit zuteilt. Bekommt er die CPU, wechselt er in den Zustand **Laufend**. Von dort kann er dreierlei tun: er läuft zu Ende und wechselt nach **Beendet**; der Scheduler entzieht ihm die CPU und setzt ihn zurück nach **Bereit**; oder er muss auf etwas warten (z. B. Tastatureingabe, Netzwerkantwort, Festplatten-I/O) und wechselt nach **Wartend / Blockiert**. Ist die Wartebedingung erfüllt, wandert er aus **Wartend** wieder nach **Bereit** — *nicht* direkt nach **Laufend**, denn erst muss der Scheduler ihm wieder eine Zeitscheibe zuteilen.

Dieser Zyklus ist der Grund, warum ein Texteditor, der auf deine Tastatureingabe wartet, keine CPU-Last erzeugt: er steht im Zustand **Wartend** und verbraucht exakt null Rechenzeit, bis du eine Taste drückst.


Das folgende Diagramm fasst die Zustandsübergänge zusammen. Die Quelle liegt als Mermaid-Datei unter [diagramme/01_prozess_lifecycle.mmd](diagramme/01_prozess_lifecycle.mmd).


In [1]:
from IPython.display import Markdown, display
with open("diagramme/01_prozess_lifecycle.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    Neu([Neu]) --> Bereit[Bereit]
    Bereit --> Laufend[Laufend]
    Laufend --> Bereit
    Laufend --> Wartend[Wartend / Blockiert]
    Wartend --> Bereit
    Laufend --> Beendet([Beendet])

```

## 2. Der Scheduler — wer darf wann auf die CPU?

Ein moderner Laptop hat typischerweise 8 oder 16 **Kerne**, aber hunderte Prozesse, die gleichzeitig „laufen" wollen. Irgendjemand muss entscheiden, welcher Prozess im aktuellen Augenblick die CPU bekommt. Diese Entscheidungen trifft der **Scheduler** — eine Komponente des Betriebssystem-Kerns.

> [!NOTE]
> Der **Scheduler** wählt aus der Menge der bereiten Prozesse denjenigen aus, der als Nächstes die CPU erhält. Die Zeit, für die ein Prozess sie erhält, nennt man eine **Zeitscheibe** (engl. *time slice* oder *quantum*). Typische Werte liegen bei 1–100 ms.

Weil das Umschalten zwischen Prozessen selbst Rechenzeit kostet (ein **Kontextwechsel** muss alle CPU-Register sichern und durch die des neuen Prozesses ersetzen), ist die Wahl der Zeitscheibe ein Balanceakt: Zu kurze Scheiben produzieren zu viel Overhead; zu lange Scheiben machen das System träge und unfair.


### Scheduling-Strategien im Überblick

In der Theorie der Betriebssysteme existieren mehrere klassische Strategien, nach denen der Scheduler seine Entscheidungen treffen kann. Jede hat ihre Stärken und Schwächen, und moderne Systeme kombinieren meist mehrere davon.

- **FCFS** (*First Come First Served*) — Prozesse werden in der Reihenfolge ihrer Ankunft bedient. Sehr einfach, aber ein einziger langer Prozess blockiert alle anderen (das sogenannte **Convoy-Problem**).
- **SJF** (*Shortest Job First*) — Der kürzeste bereite Prozess zuerst. Optimal bezüglich mittlerer Wartezeit, aber in der Praxis kaum umsetzbar, weil man die Laufzeit im Voraus kennen müsste.
- **Round-Robin (RR)** — Jeder bereite Prozess bekommt reihum eine feste Zeitscheibe. Extrem fair, typisch für interaktive Systeme wie Linux oder Windows.
- **Prioritäts-Scheduling** — Jeder Prozess trägt eine Priorität, die höchste gewinnt. Wichtig für Echtzeit-Systeme (Messdatenerfassung, Robotersteuerung); Gefahr des **Starvation**, wenn niedrige Prioritäten niemals drankommen.

In der Realität verwenden Linux (*CFS — Completely Fair Scheduler*) und Windows hybride Verfahren, die Round-Robin mit dynamisch angepassten Prioritäten kombinieren.


### Round-Robin in der Praxis

Round-Robin ist das Arbeitspferd interaktiver Betriebssysteme. Die Idee ist bestechend einfach: Alle bereiten Prozesse werden in eine Warteschlange eingereiht. Der erste bekommt die CPU für genau *eine* Zeitscheibe — zum Beispiel 10 ms. Ist er fertig oder blockiert, nimmt der nächste Prozess seinen Platz ein; ist die Zeitscheibe aber abgelaufen und der Prozess hat noch zu tun, wandert er ganz ans Ende der Warteschlange. So kommt garantiert jeder dran, egal ob fünf oder fünfhundert Prozesse warten.

> [!TIP]
> Öffne auf deinem Rechner einmal den Task-Manager (Windows) oder `htop` (Linux/macOS). Was du dort unter *CPU-Auslastung pro Prozess* siehst, ist direkt das Ergebnis der Scheduler-Entscheidungen der letzten Sekunde. Ein Prozess, der konstant bei 100 % steht, hat in jeder Zeitscheibe voll gerechnet.

> [!WARNING]
> Round-Robin garantiert *Fairness im Durchschnitt*, aber **keine Echtzeit**. Wenn du eine Robotersteuerung schreibst, die in exakt 1 ms antworten muss, reicht das nicht — dafür gibt es spezielle Echtzeit-Betriebssysteme (RTOS).


Das folgende Diagramm zeigt schematisch, wie ein Scheduler aus einer Ready-Queue nach verschiedenen Strategien auswählt. Quelle: [diagramme/03_scheduler.mmd](diagramme/03_scheduler.mmd).


In [2]:
from IPython.display import Markdown, display
with open("diagramme/03_scheduler.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    Q[Ready-Queue: P1, P2, P3, P4] --> S{Scheduler waehlt}
    S -->|FCFS| FCFS[Reihenfolge der Ankunft]
    S -->|SJF| SJF[kuerzester Job zuerst]
    S -->|RR| RR[Zeitscheibe z.B. 10ms]
    S -->|Prio| PR[hoechste Prioritaet zuerst]
    FCFS --> CPU[CPU fuehrt aus]
    SJF --> CPU
    RR --> CPU
    PR --> CPU

```

## 3. Threads — mehrere Ausführungspfade in einem Prozess

Prozesse sind teuer. Einen neuen Prozess zu erzeugen heißt, einen komplett neuen Adressraum anzulegen, alle Register zu initialisieren und das OS über den neuen Eintrag in der Prozess-Tabelle zu informieren. Wenn du innerhalb desselben Programms zwei Dinge parallel tun willst — etwa einen Sensor lesen *und* gleichzeitig die Werte ins Netz senden — lohnt sich dieser Aufwand nicht. Die leichtgewichtige Alternative heißt **Thread**.

> [!NOTE]
> Ein **Thread** (deutsch: Ausführungsfaden) ist eine parallel ausführbare Einheit innerhalb eines Prozesses. Alle Threads eines Prozesses **teilen sich denselben Adressraum** und damit auch alle Variablen und offenen Dateien. Sie haben aber jeweils einen eigenen **Stack** und eigene Register.

Threads sind der Grund, warum dein Browser gleichzeitig eine Seite rendern, einen Download verarbeiten und ein Video abspielen kann, ohne dass sich die drei Aufgaben gegenseitig blockieren. Jeder der drei läuft in einem eigenen Thread desselben Browser-Prozesses.


### Prozess vs. Thread im direkten Vergleich

| Merkmal | Prozess | Thread |
|---|---|---|
| Adressraum | eigener, isoliert | gemeinsam mit anderen Threads des Prozesses |
| Speicher pro Instanz | hoch (MB) | niedrig (KB) |
| Kommunikation untereinander | nur über OS (Pipes, Sockets, Shared Memory) | direkt über gemeinsame Variablen |
| Erzeugung | teuer | billig |
| Absturz eines Exemplars | nur dieser Prozess betroffen | ggf. **gesamter Prozess** stürzt ab |
| Typische Verwendung | unabhängige Programme | nebenläufige Teilaufgaben |

Die letzte Zeile ist der entscheidende Trade-off: Threads sind billig und schnell, aber weil sie sich Speicher teilen, können Fehler in einem Thread (etwa eine Speicher-Korruption) den gesamten Prozess zum Absturz bringen. Prozesse isolieren stärker — dafür kostet jede Inter-Prozess-Kommunikation den Umweg über das Betriebssystem.

> [!WARNING]
> Zwei Threads, die dieselbe Variable gleichzeitig ändern, können zu einer **Race Condition** führen — einem Fehler, der mal auftritt und mal nicht, je nach Scheduling-Zufall. In Python mildert der *Global Interpreter Lock* (GIL) dieses Problem ab, entfernt es aber nicht vollständig.


### Ingenieur-Beispiel: paralleler Mess-Daemon mit Threads

Ein typisches Setup an einem modernen Prüfstand: ein einziger Prozess `messdaemon.py` startet drei Threads. **Thread A** liest im Takt von 1 ms den Kraftsensor. **Thread B** puffert die Werte in einer gemeinsamen Liste und fasst sie sekundenweise zusammen. **Thread C** schickt die Zusammenfassungen alle fünf Sekunden per HTTP an einen Server im Intranet.

Weil alle drei Threads denselben Adressraum teilen, können sie direkt auf einen gemeinsamen Ring-Puffer zugreifen, ohne die Werte über einen Socket oder eine Datei umkopieren zu müssen. Für die FEM-Simulation parallel auf demselben Rechner verwendet man dagegen oft *mehrere Prozesse* (etwa mit `multiprocessing`): Hier wiegen die Isolation und die Nutzung mehrerer CPU-Kerne gleichzeitig (umgeht den GIL) den höheren Kommunikations-Overhead auf.


Das folgende Diagramm zeigt dieses Muster: ein Mess-Prozess mit mehreren Threads, die gemeinsamen Speicher nutzen — daneben ein FEM-Prozess mit eigener Speicherwelt. Quelle: [diagramme/04_prozess_vs_thread.mmd](diagramme/04_prozess_vs_thread.mmd).


In [3]:
from IPython.display import Markdown, display
with open("diagramme/04_prozess_vs_thread.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart TD
    subgraph P1[Prozess A: Messwert-Erfassung]
        T1[Thread A1: Sensor lesen]
        T2[Thread A2: Werte puffern]
        T3[Thread A3: Netzwerk senden]
        MEM1[(gemeinsamer Speicher)]
        T1 --- MEM1
        T2 --- MEM1
        T3 --- MEM1
    end
    subgraph P2[Prozess B: FEM-Simulation]
        T4[Thread B1: Rechner-Kern 1]
        T5[Thread B2: Rechner-Kern 2]
        MEM2[(eigener Speicher)]
        T4 --- MEM2
        T5 --- MEM2
    end

```

## 4. Virtueller Speicher und Paging

Wir haben bereits gesagt, dass jeder Prozess einen „eigenen Adressraum" bekommt. Das klingt zunächst, als würde das OS für jeden Prozess einen Teil des RAM reservieren. In Wirklichkeit ist die Lösung weit eleganter — und für Ingenieurinnen und Ingenieure, die mit großen Simulationsdaten arbeiten, eine essenzielle Grundlage.

> [!NOTE]
> **Virtueller Speicher** ist ein Abstraktionsmechanismus, bei dem jeder Prozess Adressen in einem fiktiven, scheinbar exklusiven Adressraum (z. B. 48 Bit = 256 TB) sieht. Das Betriebssystem und die **Memory Management Unit (MMU)** der CPU übersetzen diese virtuellen Adressen transparent in die tatsächlichen physikalischen RAM-Adressen.

Der große Vorteil: Das OS kann einzelne Teile eines Prozesses sogar auf die SSD auslagern (**Swap**), wenn der RAM knapp wird. Der Prozess merkt davon nichts — außer, dass der Zugriff dann spürbar langsamer wird.


### Paging — der Speicher wird in Seiten zerteilt

Technisch teilt das OS den virtuellen und den physikalischen Speicher in gleich große Blöcke zu typischerweise 4 KiB, genannt **Seiten** (engl. *pages*) bzw. **Rahmen** (engl. *frames*). Eine **Page Table** hält für jeden Prozess fest, welche virtuelle Seite gerade in welchem physikalischen Rahmen liegt — oder ob sie auf die SSD ausgelagert wurde.

Greift ein Prozess auf eine Adresse in einer gerade ausgelagerten Seite zu, meldet die MMU einen **Page Fault**. Das OS springt ein, lädt die fehlende Seite von der SSD in einen freien Rahmen und lässt den Prozess weiterlaufen. Dieser ganze Vorgang ist transparent — der Prozess sieht nur einen etwas langsameren Speicherzugriff.

> [!WARNING]
> Wenn zu viele Prozesse zu viel Speicher wollen und das OS ständig Seiten zwischen RAM und SSD hin- und herschaufelt, spricht man von **Thrashing**. Das System wird dabei dramatisch langsam, obwohl die CPU-Auslastung niedrig bleibt — ein klassisches Symptom, das jede Simulations-Ingenieurin schon einmal erlebt hat, wenn das FEM-Modell knapp über die RAM-Grenze hinausging.


Das folgende Diagramm zeigt den Übersetzungspfad einer virtuellen Adresse: MMU schaut in der Page Table nach, findet den Rahmen im RAM — oder triggert bei Valid=0 einen Page Fault, nach dem das OS die Seite von der SSD nachlädt. Quelle: [diagramme/05_paging.mmd](diagramme/05_paging.mmd).


In [4]:
from IPython.display import Markdown, display
with open("diagramme/05_paging.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart LR
    VA[Virtuelle Adresse: Page 3, Offset 200] --> MMU{MMU / Page Table}
    MMU -->|Valid=1| RAM[Physischer RAM: Frame 5]
    MMU -->|Valid=0 Page Fault| OS[OS laedt Seite]
    OS --> DISK[SSD / Swap]
    DISK --> RAM

```

### 4.1 Speicherhierarchie — Größenordnungen im Vergleich

Virtueller Speicher funktioniert nur deswegen gut, weil die Speicherebenen eines Rechners dramatisch unterschiedlich schnell sind. Je näher der Speicher an der CPU liegt, desto schneller ist er zugreifbar — und desto teurer und kleiner ist er.


Die Hierarchie umfasst typischerweise fünf Ebenen: **Register** im CPU-Kern (Bruchteile einer Nanosekunde), gefolgt von mehreren **Cache-Stufen** (L1 ≈ 1 ns, L2 ≈ 3 ns, L3 ≈ 10 ns), dann der **Hauptspeicher (RAM)** im Bereich von 100 ns, darunter die **SSD** bei 50 µs und ganz unten die **HDD** bei ca. 5 ms. Jede Ebene ist ungefähr um einen Faktor 10–100 langsamer als die darüber — und gleichzeitig größer. Ein L1-Cache fasst vielleicht 64 KiB, eine SSD fast beliebig viel.

Aus dieser Hierarchie folgt eine Faustregel für effizienten Code: *Wenn du etwas oft brauchst, halte es nahe an der CPU.* Das ist der eigentliche Grund, warum datenlokale Algorithmen — die nacheinander auf benachbarte Speicherstellen zugreifen — so viel schneller sind als sprunghafte. Cache-Effekte können einen FEM-Löser um Faktor 5 bis 10 beschleunigen, ohne dass der Algorithmus inhaltlich geändert wurde.

Quelle des Diagramms: [diagramme/06_speicherhierarchie_os.mmd](diagramme/06_speicherhierarchie_os.mmd).


In [5]:
from IPython.display import Markdown, display
with open("diagramme/06_speicherhierarchie_os.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart TD
    CPU[CPU] --> REG[Register ns]
    REG --> L1[L1-Cache ~1 ns]
    L1 --> L2[L2-Cache ~3 ns]
    L2 --> L3[L3-Cache ~10 ns]
    L3 --> RAM[RAM ~100 ns]
    RAM --> SSD[SSD ~50 us]
    SSD --> HDD[HDD ~5 ms]

```

## 5. Das Dateisystem

Die CPU-Konzepte waren der eine Aufgabenbereich des Betriebssystems. Der andere, für dich als Anwender mindestens so sichtbare Bereich ist das **Dateisystem**. Hier kümmert sich das OS darum, dass aus dem flachen Byte-Strom einer SSD oder Festplatte eine hierarchische Struktur aus Ordnern und Dateien wird, und dass Zugriffsrechte durchgesetzt werden.


### 5.1 Baum-Struktur

Jedes moderne Dateisystem ist ein **Baum**. An der Spitze steht ein einzelner Wurzelknoten (`/` unter Linux/macOS, `C:\` unter Windows). Unterhalb liegen **Verzeichnisse** (auch *Ordner*), die wiederum Verzeichnisse oder **Dateien** enthalten. Dateien sind Blätter des Baums, Verzeichnisse innere Knoten.

Ein typischer Pfad wie `/home/student/fertigung/messdaten.csv` ist nichts anderes als eine Wegbeschreibung durch diesen Baum, beginnend an der Wurzel. Unter Windows mit `C:\Users\Anna\messung.csv` funktioniert das gleich — nur mit Backslashes und einem laufwerksspezifischen Wurzelknoten.

> [!TIP]
> Python glättet diese Windows-vs-Linux-Unterschiede für dich, wenn du **`os.path.join`** oder (moderner) **`pathlib.Path`** benutzt. Niemals solltest du Pfade per String-Konkatenation zusammenbauen — das bricht spätestens beim ersten Plattformwechsel.


### 5.2 Zugriffsrechte: das rwx-Modell

Unter Unix-artigen Systemen trägt jede Datei und jedes Verzeichnis drei Gruppen von **Rechten**: Lesen (**r**), Schreiben (**w**) und Ausführen (**x**). Die drei Rechte werden getrennt für *Besitzer*, *Gruppe* und *alle anderen* festgelegt. Eine Datei mit den Rechten `rwxr-xr--` ist vom Besitzer voll nutzbar, von seiner Gruppe les- und ausführbar, von allen anderen nur lesbar.

Für *Verzeichnisse* bedeuten die Rechte etwas leicht anderes: `r` erlaubt das *Auflisten* des Inhalts, `w` das Anlegen und Löschen von Einträgen, `x` das *Betreten*. Ein Verzeichnis ohne `x`-Recht ist wie ein verschlossener Raum — man sieht vielleicht, was drin steht, aber kann nicht hineingehen.

Unter Windows existiert ein ähnliches, aber feingranulareres Modell über **Access Control Lists (ACLs)**, das aber denselben Grundgedanken umsetzt: *Wer darf was mit welchem Objekt tun?*

Quelle des Baum-Diagramms: [diagramme/02_dateisystem_baum.mmd](diagramme/02_dateisystem_baum.mmd).


In [6]:
from IPython.display import Markdown, display
with open("diagramme/02_dateisystem_baum.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))


```mermaid
flowchart TD
    root[/ Wurzel/]
    root --> home[home/]
    root --> var[var/]
    root --> etc[etc/]
    home --> user[student/]
    user --> proj[fertigung/]
    proj --> csv[messdaten.csv]
    proj --> py[analyse.py]
    proj --> log[logs/]
    log --> l1[tag01.log]
    log --> l2[tag02.log]

```

## 6. Zwei Ingenieur-Beispiele zum Einordnen

Um die bisher abstrakt erklärten Konzepte fest zu verankern, schauen wir uns zwei sehr reale Szenarien aus dem Maschinenbau-Alltag an. In beiden treffen Prozesse, Threads, Scheduler und Dateisystem zusammen.


### Beispiel 1: Datenlogger als Hintergrund-Prozess

Stell dir vor, du willst einen Temperatursensor über 24 Stunden aufzeichnen. Du startest dein Python-Skript `datenlogger.py` und willst danach den Rechner *trotzdem* weiter normal benutzen können. Das Skript wird dafür als eigener **Prozess** gestartet, typischerweise mit niedriger Priorität. Der **Scheduler** gibt ihm regelmäßig kleine Zeitscheiben, damit er alle Sekunde eine Messung machen kann. Die CPU-Auslastung bleibt dabei praktisch bei 0 %, weil der Prozess die meiste Zeit im Zustand **Wartend** (auf den Sensor-Timer) ruht.

Der Datenlogger öffnet eine Logdatei im Dateisystem und schreibt regelmäßig neue Zeilen. Weil die Datei vom Betriebssystem gepuffert wird, geschieht die eigentliche Festplatten-Schreibung nur alle paar Kilobyte — ein schönes Beispiel dafür, wie das OS im Hintergrund optimiert, ohne dass dein Code etwas davon mitbekommt.

Wird zwischendrin der Arbeitsspeicher knapp, weil du parallel ein Video schaust und der RAM fast voll ist, wandern Teile deines Logger-Speichers möglicherweise per **Paging** auf die SSD. Beim nächsten Zugriff entsteht ein kurzer **Page Fault**, und die Seite kommt zurück in den RAM. Der Logger merkt davon nichts — außer einer minimalen Verzögerung.


### Beispiel 2: FEM-Simulation auf vielen Kernen

Nun der Gegenpol: eine Finite-Elemente-Simulation mit 10 Millionen Knoten. Hier willst du *maximale* CPU-Last — und zwar auf allen 16 Kernen gleichzeitig. Typisch startet der Solver **mehrere Prozesse** (oft einen pro Kern) und teilt das Gitter zwischen ihnen auf. Innerhalb jedes Prozesses laufen wiederum **Threads**, die stark parallelisierte Matrizen-Operationen ausführen.

Der **Scheduler** hat es hier einfach: so viele Prozesse wie Kerne, alle rechnen, keine wartet. Kritisch wird der **Speicher**: 10 Millionen Knoten mit je ein paar Freiheitsgraden belegen schnell 20 GiB RAM. Passt das nicht mehr rein, setzt **Thrashing** ein — und deine Simulation wird um Faktor 100 langsamer. Die wichtige Lehre aus dieser Beobachtung: kenne deine RAM-Grenze, bevor du den Solver startest.

Die Zwischenergebnisse landen währenddessen im Dateisystem, oft in einem dedizierten Arbeitsverzeichnis mit entsprechenden **rwx-Rechten** — schließlich soll niemand sonst auf deinem Rechencluster in deine Zwischendateien schauen können.


## 7. Selbst-Check

Prüfe dein Verständnis mit den folgenden Fragen. Klicke auf „Antwort anzeigen", um die Lösung zu sehen — aber erst, nachdem du selbst eine Antwort formuliert hast.


<details>
<summary><b>Frage 1:</b> Worin besteht der fundamentale Unterschied zwischen einem Programm und einem Prozess?</summary>

Ein Programm ist eine *statische Datei* auf der Festplatte — Bytecode, der ausgeführt werden *könnte*. Ein Prozess ist eine konkrete *laufende Instanz* dieses Programms im Arbeitsspeicher, mit eigener PID, eigenem Adressraum, eigenem Stack, offenen Datei-Handles und eigenem Zustand im Scheduler.
</details>

<details>
<summary><b>Frage 2:</b> In welchen Zustand wechselt ein Prozess, der gerade auf eine Tastatureingabe wartet — und warum verbraucht er dabei keine CPU?</summary>

Er wechselt in den Zustand **Wartend / Blockiert**. Der Scheduler vergibt CPU-Zeit nur an Prozesse im Zustand **Bereit**. Ein wartender Prozess wird vom Scheduler bewusst übergangen, bis die Wartebedingung erfüllt ist (Eingabe kommt an), worauf ihn das OS zurück nach *Bereit* setzt.
</details>

<details>
<summary><b>Frage 3:</b> Warum ist Round-Robin für einen Echtzeit-Regelkreis mit 1 ms Antwortzeit nicht geeignet?</summary>

Round-Robin garantiert Fairness, aber **keine obere Schranke für die Antwortzeit**. Sind z. B. 50 andere Prozesse bereit, kann es 50 Zeitscheiben dauern, bis unser Regler wieder die CPU bekommt. Für harte Echtzeitbedingungen braucht man einen RTOS-Scheduler mit festen Prioritäten und deterministischen Antwortzeiten.
</details>

<details>
<summary><b>Frage 4:</b> Zwei Threads eines Prozesses arbeiten parallel auf derselben Liste. Welche Gefahr droht, und was hat sie mit dem gemeinsamen Adressraum zu tun?</summary>

Es droht eine **Race Condition**. Weil beide Threads denselben Adressraum und damit dieselbe Liste sehen, können gleichzeitige Schreibzugriffe zu inkonsistenten Zwischenzuständen führen — z. B. verlorene Einträge oder korrumpierte interne Datenstrukturen. Man braucht Synchronisations-Primitive wie Locks, um den kritischen Bereich zu schützen.
</details>

<details>
<summary><b>Frage 5:</b> Was passiert technisch bei einem Page Fault, und warum ist er <i>meistens</i> kein Fehler?</summary>

Bei einem Page Fault versucht ein Prozess auf eine virtuelle Seite zuzugreifen, die momentan nicht im physikalischen RAM liegt. Die MMU löst eine Ausnahme aus, das OS übernimmt, lädt die Seite von der SSD (oder initialisiert sie neu, falls sie noch nie gelesen wurde) in einen freien Rahmen und setzt den Prozess fort. Das ist kein Fehler im Programmiersinn, sondern ein normaler Mechanismus des virtuellen Speichers. Nur wenn die Seite überhaupt nicht existieren darf (Null-Pointer, Zugriff außerhalb des Adressraums), ist es ein echter Absturz-Grund.
</details>

<details>
<summary><b>Frage 6:</b> Warum ist ein Zugriff auf den L1-Cache ungefähr 100-mal schneller als auf den RAM, obwohl beide „elektronischer Speicher" sind?</summary>

L1-Caches sitzen *direkt im CPU-Kern* und sind mit SRAM-Zellen gebaut, die in wenigen Takten ansprechbar sind. Der RAM-Chip hängt am Speicherbus, muss Zeilen in DRAM-Matrizen aktivieren, und die Signallaufzeit über Platinen-Leiterbahnen ist bereits physikalisch nicht unter ein paar Nanosekunden zu bringen. Die Hierarchie gibt es genau deswegen: schnell muss nahe sein, nahe muss klein sein.
</details>

<details>
<summary><b>Frage 7:</b> Ein Ordner hat die Rechte <code>r-xr-x---</code>. Was darfst du als <i>nicht</i>-Besitzer, aber Mitglied der Gruppe, damit tun?</summary>

Du darfst den Ordner **betreten** (x) und den Inhalt **auflisten** (r), aber keine Dateien darin anlegen, umbenennen oder löschen (das wäre **w**). Einzelne Dateien darin könntest du durchaus lesen — vorausgesetzt, die Dateien selbst gewähren das entsprechende Recht.
</details>

<details>
<summary><b>Frage 8:</b> Warum startet ein FEM-Solver typischerweise mehrere <i>Prozesse</i> und nicht nur viele Threads innerhalb eines Prozesses?</summary>

In Python gibt es den **GIL** (Global Interpreter Lock), der verhindert, dass mehrere Threads gleichzeitig Python-Bytecode ausführen — bei reiner CPU-Last bringt Threading dann keinen Gewinn auf mehreren Kernen. Auch in nativen Sprachen ist die Prozess-Isolation für rechenlastige, unabhängige Arbeits-Pakete oft robuster: ein Absturz eines Prozesses beeinträchtigt nicht die anderen, und das OS kann sie sauber auf verschiedene Kerne verteilen.
</details>


## ✅ Zusammenfassung

- Ein **Prozess** ist eine in Ausführung befindliche Programm-Instanz mit eigenem, isoliertem Adressraum und eigener PID.
- Prozesse durchlaufen den Lebenszyklus *Neu → Bereit → Laufend → Wartend → Beendet* unter Regie des **Schedulers**.
- Gängige Scheduling-Strategien sind **FCFS**, **SJF**, **Round-Robin** und **Prioritäts-Scheduling**; moderne Systeme kombinieren diese.
- **Threads** sind leichtgewichtige Ausführungspfade innerhalb eines Prozesses; sie teilen sich den Adressraum und damit alle Variablen — mit allen Vor- und Nachteilen, die daraus folgen.
- **Virtueller Speicher** und **Paging** verschaffen jedem Prozess den Anschein, den gesamten Speicher für sich zu haben. Ein **Page Fault** lädt fehlende Seiten von der SSD.
- Die **Speicherhierarchie** (Register < Cache < RAM < SSD < HDD) bestimmt die Laufzeit realer Programme oft stärker als der Algorithmus selbst.
- Das **Dateisystem** ist ein Baum aus Ordnern und Dateien; **rwx-Rechte** (oder ACLs unter Windows) regeln, wer was darf.

## ➡️ Nächster Schritt
Weiter mit [02_praxis.ipynb](02_praxis.ipynb) — dort lernst du, mit dem `os`-Modul in Python das Dateisystem praktisch zu bearbeiten.
